# Sequences, Longest Increasing Subsequences, and the Erdős–Szekeres Theorem

**A-Level Mathematics Extension: Combinatorics and the Pigeonhole Principle**

---

## Welcome!

Imagine you have a sequence of numbers, like the daily closing prices of a stock
over a month. You want to find the longest period during which prices were
consistently rising. This is the **Longest Increasing Subsequence** (LIS) problem.

But there is a deeper question: in any sufficiently long sequence, can you
*always* guarantee a long monotone (increasing or decreasing) subsequence?

The answer — the **Erdős–Szekeres Theorem** — is one of the most elegant results
in combinatorics, with a proof so short it fits on one line!

## Section 1: Introduction — Sequences and Subsequences

### Definitions

A **sequence** is an ordered list of numbers: $a_1, a_2, \ldots, a_n$.

A **subsequence** is obtained by deleting some elements (possibly none)
without changing the order of the remaining elements.

For example, from $(3, 1, 4, 1, 5, 9, 2, 6)$:
- $(3, 4, 5, 9)$ is an **increasing** subsequence (each term larger than the previous)
- $(4, 1)$ is a **decreasing** subsequence
- $(1, 1, 2)$ is also a valid subsequence (but not strictly increasing)

### Key Problems

- **LIS** (Longest Increasing Subsequence): find the longest *strictly* increasing subsequence.
- **LDS** (Longest Decreasing Subsequence): find the longest *strictly* decreasing subsequence.

### The Erdős–Szekeres Theorem

> **Theorem (1935)**: Any sequence of more than $mn$ distinct real numbers
> contains either an increasing subsequence of length $m+1$
> or a decreasing subsequence of length $n+1$.

**Corollary**: Any sequence of $n^2 + 1$ distinct numbers contains a
monotone subsequence of length $n+1$.

The proof uses the **Pigeonhole Principle** and is beautifully short.
We will discover it through computation.

## Section 2: Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import permutations

np.random.seed(2024)
print('Libraries loaded!')

## Section 3: Patience Sorting

**Patience sorting** is a card-sorting algorithm that also happens to compute
the length of the LIS. Here are the rules:

1. Start with an empty table of piles.
2. Process each card (number) from left to right.
3. Place it on the **leftmost pile** whose top card is $\geq$ the current card.
4. If no such pile exists, start a **new pile** on the right.

**Key fact**: The number of piles at the end equals the **length of the LIS**!

This is a beautiful connection between a card game and combinatorics.

In [ ]:
def patience_sort(seq):
    """
    Perform patience sorting on a sequence.

    Each element is placed on the leftmost pile whose top >= element.
    If no such pile exists, a new pile is started.

    Returns
    -------
    piles : list of lists
        Each inner list is a pile (index 0 = bottom, -1 = top).
        The number of piles equals the LIS length.
    """
    piles = []  # List of piles; pile[j][-1] is the top card of pile j
    for card in seq:
        placed = False
        for pile in piles:
            if pile[-1] >= card:  # Can place on this pile
                pile.append(card)
                placed = True
                break
        if not placed:
            piles.append([card])  # Start a new pile
    return piles


def lis_length(seq):
    """Return the length of the LIS of seq."""
    return len(patience_sort(seq))


def find_lis(seq):
    """
    Find one longest increasing subsequence of seq using patience sorting
    with backtracking.

    Algorithm:
    1. Perform patience sorting, recording which pile each element goes to.
    2. Work backwards through the sequence: greedily pick the rightmost element
       in the last pile, then the second-to-last pile, etc.

    Returns
    -------
    list
        One longest strictly increasing subsequence.
    """
    piles = []
    pile_idx = []  # Which pile did element i go to?

    for i, card in enumerate(seq):
        placed = False
        for j, pile in enumerate(piles):
            if pile[-1] >= card:
                pile.append(card)
                pile_idx.append(j)  # Recorded: element i went to pile j
                placed = True
                break
        if not placed:
            piles.append([card])
            pile_idx.append(len(piles) - 1)  # New pile index

    # Reconstruction: work backwards from the last pile
    k = len(piles)   # Number of piles = LIS length
    lis = []
    last_pile = k - 1  # We want one element from each pile, starting from the last

    for i in range(len(seq) - 1, -1, -1):
        if pile_idx[i] == last_pile:
            lis.append(seq[i])
            last_pile -= 1
            if last_pile < 0:
                break

    return list(reversed(lis))  # Reverse to get increasing order


# ---- Test on a simple example ----
seq_test = [3, 1, 4, 2, 8, 3, 7, 6]
piles_test = patience_sort(seq_test)
lis_test = find_lis(seq_test)

print(f'Sequence: {seq_test}')
print(f'Piles after patience sorting:')
for j, pile in enumerate(piles_test):
    print(f'  Pile {j+1}: {pile}  (top = {pile[-1]})')
print(f'Number of piles = LIS length = {len(piles_test)}')
print(f'One LIS: {lis_test}')
print()

# Verify it's increasing
is_increasing = all(lis_test[i] < lis_test[i+1] for i in range(len(lis_test)-1))
print(f'Is the LIS strictly increasing? {is_increasing}')

## Section 4: Visualising Patience Sorting Step by Step

Let's animate the patience sorting process on our sequence $(3,1,4,2,8,3,7,6)$.
At each step we'll show the current pile configuration as a bar chart.

In [ ]:
def visualise_patience_sort(seq):
    """
    Show the pile configuration after each card is placed,
    as a grid of subplots.
    """
    n = len(seq)
    cols = 4
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = np.array(axes).flatten()

    piles = []  # Build up the piles step by step

    for step, card in enumerate(seq):
        ax = axes[step]

        # Place the card (same logic as patience_sort)
        placed = False
        new_pile_idx = None
        for j, pile in enumerate(piles):
            if pile[-1] >= card:
                pile.append(card)
                placed = True
                new_pile_idx = j
                break
        if not placed:
            piles.append([card])
            new_pile_idx = len(piles) - 1

        # Draw the current pile tops as bars
        tops = [pile[-1] for pile in piles]
        pile_sizes = [len(pile) for pile in piles]

        bar_colors = ['tomato' if j == new_pile_idx else 'steelblue'
                      for j in range(len(piles))]

        bars = ax.bar(range(len(piles)), tops, color=bar_colors, alpha=0.8,
                      edgecolor='black', linewidth=1.2)

        # Annotate bars with the top card value and pile depth
        for j, (bar, top, size) in enumerate(zip(bars, tops, pile_sizes)):
            ax.text(bar.get_x() + bar.get_width()/2, top + 0.1,
                    f'{top}\n({size})', ha='center', va='bottom', fontsize=9)

        ax.set_title(f'Step {step+1}: place {card}', fontsize=10)
        ax.set_xlabel('Pile index', fontsize=8)
        ax.set_ylabel('Top card value', fontsize=8)
        ax.set_ylim(0, max(seq) + 1.5)
        ax.set_xticks(range(len(piles)))
        ax.set_xticklabels([f'P{j+1}' for j in range(len(piles))])
        ax.grid(axis='y', linestyle='--', alpha=0.5)

    # Hide unused subplots
    for i in range(n, len(axes)):
        axes[i].set_visible(False)

    # Legend
    red_patch  = mpatches.Patch(color='tomato',    label='Newly placed card')
    blue_patch = mpatches.Patch(color='steelblue', label='Existing pile top')
    fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=10)

    fig.suptitle(f'Patience Sorting: {seq}\n(Bar = top card value, number in parentheses = pile depth)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Final number of piles = {len(piles)} = LIS length')


visualise_patience_sort([3, 1, 4, 2, 8, 3, 7, 6])

### What to Notice

- Each pile's top card is always **decreasing from left to right**.
  (If this weren't true, we'd have placed the card differently.)
- When we start a new pile, it means the new card is **larger than all current pile tops** —
  which is exactly when the LIS grows.
- The **top cards of the piles** at the end form the longest increasing subsequence
  (or can be used to reconstruct it via backtracking).

## Section 5: Visualising the LIS

Let's plot the full sequence as a bar chart and highlight the elements
belonging to the LIS in a different colour.

In [ ]:
def visualise_lis(seq, title=None):
    """
    Plot a sequence as a bar chart and highlight the LIS elements.
    """
    lis = find_lis(seq)
    lis_length_val = len(lis)

    # Identify which positions are in the LIS
    # We match left-to-right: find the first occurrence of each LIS element
    lis_positions = set()
    lis_copy = list(lis)  # Make a copy to consume
    lis_ptr = 0
    for i, val in enumerate(seq):
        if lis_ptr < len(lis_copy) and val == lis_copy[lis_ptr]:
            lis_positions.add(i)
            lis_ptr += 1

    colors = ['tomato' if i in lis_positions else 'steelblue' for i in range(len(seq))]

    fig, ax = plt.subplots(figsize=(max(8, len(seq)), 4))
    bars = ax.bar(range(len(seq)), seq, color=colors, edgecolor='black',
                  alpha=0.85, linewidth=1.2)

    # Annotate each bar with its value
    for i, (bar, val) in enumerate(zip(bars, seq)):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.15,
                str(val), ha='center', va='bottom', fontsize=11,
                fontweight='bold' if i in lis_positions else 'normal')

    # Draw arrows connecting the LIS elements
    lis_pos_list = sorted(lis_positions)
    for k in range(len(lis_pos_list) - 1):
        i, j = lis_pos_list[k], lis_pos_list[k+1]
        ax.annotate('',
                    xy=(j, seq[j] + 0.5),
                    xytext=(i, seq[i] + 0.5),
                    arrowprops=dict(arrowstyle='->', color='darkred',
                                   lw=2, connectionstyle='arc3,rad=-0.3'))

    ax.set_xticks(range(len(seq)))
    ax.set_xticklabels([f'a[{i}]={seq[i]}' for i in range(len(seq))],
                       rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Value', fontsize=12)

    if title is None:
        title = f'Sequence: {list(seq)}'
    ax.set_title(f'{title}\nLIS = {lis}  (length = {lis_length_val})', fontsize=12)

    red_patch  = mpatches.Patch(color='tomato',    label=f'LIS elements: {lis}')
    blue_patch = mpatches.Patch(color='steelblue', label='Other elements')
    ax.legend(handles=[red_patch, blue_patch], fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()


# Visualise the LIS for our main example
visualise_lis([3, 1, 4, 2, 8, 3, 7, 6], title='Sequence (3, 1, 4, 2, 8, 3, 7, 6)')

# Also check a few more examples
for seq in [[5, 4, 3, 2, 1], [1, 2, 3, 4, 5], [2, 3, 1, 4, 0, 5, 7, 6]]:
    lis = find_lis(seq)
    print(f'seq={seq}  ->  LIS={lis}  (length={len(lis)})')

## Section 6: Erdős–Szekeres Theorem Demonstration

The theorem says: any sequence of more than $n^2$ distinct numbers has
an increasing subsequence of length $> n$ OR a decreasing subsequence of length $> n$.

For $n = 3$: any sequence of $3^2 + 1 = 10$ elements contains a monotone
subsequence of length $4$.

Let's test this empirically on 100 random permutations of $\{1, \ldots, 10\}$.

In [ ]:
def lds_length(seq):
    """
    Length of the Longest Decreasing Subsequence (LDS).
    Computed as the LIS of the reversed sequence.
    """
    # LDS of seq = LIS of (-seq) = LIS of seq in reversed order
    return lis_length([-x for x in seq])


def find_lds(seq):
    """Find one longest decreasing subsequence."""
    neg_lis = find_lis([-x for x in seq])
    return [-x for x in neg_lis]


# Erdős–Szekeres: n=3, so sequences of length 10 = 3^2 + 1
# must have LIS >= 4 or LDS >= 4.

n_param = 3              # The 'n' in the theorem
seq_len = n_param**2 + 1 # 10 elements
threshold = n_param + 1  # Must find monotone subseq of length >= 4

num_trials = 100
np.random.seed(42)

results = []
all_have_monotone = True

for trial in range(num_trials):
    perm = np.random.permutation(seq_len).tolist()
    l = lis_length(perm)
    d = lds_length(perm)
    has_monotone = (l >= threshold) or (d >= threshold)
    results.append({'perm': perm, 'lis': l, 'lds': d, 'ok': has_monotone})
    if not has_monotone:
        all_have_monotone = False
        print(f'COUNTEREXAMPLE FOUND (this should NEVER happen): {perm}')

lis_values = [r['lis'] for r in results]
lds_values = [r['lds'] for r in results]
max_mono   = [max(r['lis'], r['lds']) for r in results]

print(f'Erdős–Szekeres theorem check:')
print(f'  Sequences of length {seq_len} = {n_param}² + 1')
print(f'  Must have LIS ≥ {threshold} OR LDS ≥ {threshold}')
print(f'  Trials: {num_trials}')
print(f'  All had a monotone subseq of length ≥ {threshold}? {all_have_monotone}')
print()
print(f'  LIS lengths:  min={min(lis_values)}, avg={np.mean(lis_values):.2f}, max={max(lis_values)}')
print(f'  LDS lengths:  min={min(lds_values)}, avg={np.mean(lds_values):.2f}, max={max(lds_values)}')
print(f'  max(LIS,LDS): min={min(max_mono)}, avg={np.mean(max_mono):.2f}, max={max(max_mono)}')

# ---- Plot distribution of LIS and LDS lengths ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, label, color in [
    (axes[0], lis_values, 'LIS length', 'steelblue'),
    (axes[1], lds_values, 'LDS length', 'tomato'),
]:
    counts = np.bincount(vals)
    ax.bar(range(len(counts)), counts, color=color, alpha=0.8, edgecolor='black')
    ax.axvline(x=threshold - 0.5, color='black', linestyle='--', linewidth=2,
               label=f'Threshold = {threshold}')
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Distribution of {label}\nacross {num_trials} random permutations of length {seq_len}',
                 fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle(f'Erdős–Szekeres: Every length-{seq_len} permutation has LIS≥{threshold} or LDS≥{threshold}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### What the Histograms Show

Every single one of our 100 random permutations of length 10 had either
LIS $\geq 4$ or LDS $\geq 4$ (or both!). The theorem is verified empirically.

Notice that many permutations have both a long LIS AND a long LDS — the
theorem only guarantees *at least one*, but reality is often more generous.

## Section 7: Tight Examples — The Grid Construction

The Erdős–Szekeres theorem is **tight**: for every $n$, there exist sequences
of length exactly $n^2$ that have no monotone subsequence of length $n+1$.

**Construction**: Arrange numbers in $n$ decreasing runs of length $n$:
$$n, n-1, \ldots, 1, \; 2n, 2n-1, \ldots, n+1, \; 3n, 3n-1, \ldots, 2n+1, \; \ldots$$

For $n = 3$: $(3, 2, 1, 6, 5, 4, 9, 8, 7)$ — length $9 = 3^2$.
- LIS: pick one from each run, e.g. $(1, 4, 7)$ or $(3, 6, 9)$ — length exactly 3.
- LDS: pick within one run, e.g. $(3, 2, 1)$ — length exactly 3.
- No monotone subsequence of length 4 exists!

In [ ]:
def tight_example(n):
    """
    Construct the tight example for Erdős–Szekeres:
    n decreasing runs of length n, giving a sequence of length n^2
    with no monotone subsequence of length n+1.
    """
    seq = []
    for run in range(n):
        # Run number 'run' contains: (run+1)*n, (run+1)*n - 1, ..., run*n + 1
        for k in range(n, 0, -1):
            seq.append(run * n + k)
    return seq


# Test for n = 1, 2, 3, 4
print('Tight examples for Erdős–Szekeres theorem:')
print(f'{"n":<6} {"n²":<6} {"Sequence":<40} {"LIS":<6} {"LDS":<6}')
print('-' * 70)

for n in [1, 2, 3, 4]:
    seq = tight_example(n)
    l = lis_length(seq)
    d = lds_length(seq)
    seq_str = str(seq)
    if len(seq_str) > 38:
        seq_str = seq_str[:35] + '...'
    print(f'{n:<6} {n**2:<6} {seq_str:<40} {l:<6} {d:<6}')

print()
print('Observation: LIS = LDS = n for each tight example.')
print('The Erdős–Szekeres theorem says we need n² + 1 elements to guarantee')
print('a monotone subsequence of length n + 1. The tight example shows n²')
print('is NOT enough — making the bound sharp!')

# Visualise the n=4 tight example
seq_tight = tight_example(4)
lis_tight = find_lis(seq_tight)
lds_tight = find_lds(seq_tight)
print()
print(f'n=4 tight example: {seq_tight}')
print(f'  LIS = {lis_tight}  (length = {len(lis_tight)})')
print(f'  LDS = {lds_tight}  (length = {len(lds_tight)})')

# Plot the tight example
fig, ax = plt.subplots(figsize=(12, 4))
n_param = 4
seq = tight_example(n_param)
lis_4 = find_lis(seq)

# Identify LIS positions
lis_pos = set()
lis_ptr = 0
for i, val in enumerate(seq):
    if lis_ptr < len(lis_4) and val == lis_4[lis_ptr]:
        lis_pos.add(i)
        lis_ptr += 1

colors = ['tomato' if i in lis_pos else 'steelblue' for i in range(len(seq))]

# Shade the runs
for run in range(n_param):
    start = run * n_param
    end   = start + n_param
    ax.axvspan(start - 0.5, end - 0.5, alpha=0.1,
               color='green' if run % 2 == 0 else 'purple')

ax.bar(range(len(seq)), seq, color=colors, edgecolor='black', alpha=0.85)
for i, val in enumerate(seq):
    ax.text(i, val + 0.15, str(val), ha='center', va='bottom', fontsize=10,
            fontweight='bold' if i in lis_pos else 'normal')

ax.set_xticks(range(len(seq)))
ax.set_xticklabels([f'{v}' for v in seq])
ax.set_title(f'Tight example for n={n_param}: sequence of length {n_param}²={n_param**2}\n'
             f'LIS length = {len(lis_4)} = n   (no monotone subseq of length n+1={n_param+1}!)',
             fontsize=12)

red_patch  = mpatches.Patch(color='tomato',    label=f'LIS = {lis_4}')
blue_patch = mpatches.Patch(color='steelblue', label='Other elements')
green_patch = mpatches.Patch(color='green', alpha=0.3, label='Decreasing runs')
ax.legend(handles=[red_patch, blue_patch, green_patch], fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Section 8: The (eᵢ, dᵢ) Labelling and the Pigeonhole Proof

Here is the most elegant part — the actual **proof** of the Erdős–Szekeres theorem,
which uses the Pigeonhole Principle.

### The Labelling

For each position $i$ in a sequence $a_1, a_2, \ldots, a_N$, define:
- $e_i$ = length of the longest **increasing** subsequence **ending at** $a_i$
- $d_i$ = length of the longest **decreasing** subsequence **ending at** $a_i$

**Key fact**: All pairs $(e_i, d_i)$ are **distinct** for any sequence of distinct numbers!

**Proof sketch**: If $i < j$ and $a_i < a_j$, then $e_j > e_i$ (we can extend
the increasing subsequence ending at $a_i$ by appending $a_j$). Similarly for decreasing.
This forces all pairs to be different.

### The Pigeonhole Argument

If all pairs $(e_i, d_i)$ satisfy $e_i \leq m$ and $d_i \leq n$, then there
are only $mn$ possible distinct pairs — but we have $mn + 1$ elements!
By the Pigeonhole Principle, two elements must share a pair, a contradiction.

Therefore, some element must have $e_i \geq m+1$ or $d_i \geq n+1$. QED.

Let's compute and display this labelling.

In [ ]:
def compute_ed_labels(seq):
    """
    Compute (e_i, d_i) labels for each position in seq.

    e_i = length of LIS ending at position i
    d_i = length of LDS ending at position i

    Uses O(n^2) dynamic programming.
    """
    n = len(seq)
    e = [1] * n  # e[i] = LIS length ending at i
    d = [1] * n  # d[i] = LDS length ending at i

    for i in range(n):
        for j in range(i):
            if seq[j] < seq[i]:  # Can extend increasing subseq
                e[i] = max(e[i], e[j] + 1)
            if seq[j] > seq[i]:  # Can extend decreasing subseq
                d[i] = max(d[i], d[j] + 1)

    return list(zip(e, d))


# ---- Demonstrate on a small example ----
example_seq = [5, 3, 4, 1, 2, 8, 6, 7]
labels = compute_ed_labels(example_seq)

print('Sequence:', example_seq)
print()
print(f'  i  |  a_i  |  e_i (LIS len ending here)  |  d_i (LDS len ending here)  |  (e_i, d_i)')
print('  ' + '-' * 65)
for i, (val, (e, d)) in enumerate(zip(example_seq, labels)):
    print(f'  {i}  |   {val}   |            {e}              |            {d}              |  ({e}, {d})')

print()
# Check all pairs are distinct
pairs = [labels[i] for i in range(len(example_seq))]
all_distinct = len(pairs) == len(set(pairs))
print(f'All (e_i, d_i) pairs distinct? {all_distinct}')
print(f'Pairs: {pairs}')

print()
print(f'Max e_i = {max(e for e,d in labels)} (= LIS length of whole sequence)')
print(f'Max d_i = {max(d for e,d in labels)} (= LDS length of whole sequence)')

In [ ]:
def visualise_ed_labels(seq):
    """
    Visualise the (e_i, d_i) labelling for a sequence.
    Each element is shown as a point in the (e, d) plane.
    """
    labels = compute_ed_labels(seq)
    es = [e for e, d in labels]
    ds = [d for e, d in labels]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: the sequence as a bar chart annotated with labels
    ax = axes[0]
    ax.bar(range(len(seq)), seq, color='steelblue', alpha=0.7, edgecolor='black')
    for i, (val, (e, d)) in enumerate(zip(seq, labels)):
        ax.text(i, val + 0.2, f'({e},{d})', ha='center', va='bottom', fontsize=9, color='darkred')
        ax.text(i, -0.8, f'{val}', ha='center', va='top', fontsize=9)
    ax.set_xticks(range(len(seq)))
    ax.set_xticklabels([f'$a_{{{i}}}$' for i in range(len(seq))])
    ax.set_ylabel('Value', fontsize=12)
    ax.set_title(f'Sequence {seq}\nLabels $(e_i, d_i)$ shown above each bar', fontsize=11)
    ax.set_ylim(-1.5, max(seq) + 2)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    # Right: scatter plot of (e_i, d_i) pairs
    ax2 = axes[1]
    scatter_colors = plt.cm.tab10(np.linspace(0, 1, len(seq)))
    for i, (e, d, col) in enumerate(zip(es, ds, scatter_colors)):
        ax2.scatter(e, d, color=col, s=150, zorder=5)
        ax2.annotate(f'$a_{i}$={seq[i]}', (e, d),
                     textcoords='offset points', xytext=(6, 3), fontsize=9)

    # Draw the grid of possible (e,d) pairs
    max_e = max(es) + 1
    max_d = max(ds) + 1
    for e_val in range(1, max_e + 1):
        for d_val in range(1, max_d + 1):
            ax2.scatter(e_val, d_val, color='lightgray', s=60, zorder=1)

    ax2.set_xlabel('$e_i$ (LIS ending at position $i$)', fontsize=12)
    ax2.set_ylabel('$d_i$ (LDS ending at position $i$)', fontsize=12)
    ax2.set_title('$(e_i, d_i)$ pairs in the grid\n(All pairs are DISTINCT — key to the proof!)',
                  fontsize=11)
    ax2.set_xticks(range(1, max_e + 1))
    ax2.set_yticks(range(1, max_d + 1))
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')

    plt.suptitle('The (eᵢ, dᵢ) Labelling: Heart of the Erdős–Szekeres Proof',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Verify distinctness
    all_distinct = len(labels) == len(set(labels))
    print(f'All {len(labels)} pairs (eᵢ, dᵢ) are distinct: {all_distinct}')
    print(f'Max grid size needed: {max(es)} × {max(ds)} = {max(es)*max(ds)}')
    print(f'If LIS ≤ {max(es)} and LDS ≤ {max(ds)}, we need ≤ {max(es)*max(ds)} positions.')
    print(f'Our sequence has {len(seq)} positions — which fits.')


visualise_ed_labels([5, 3, 4, 1, 2, 8, 6, 7])

### The Pigeonhole Contradiction

Looking at the right plot above:

- All $(e_i, d_i)$ pairs land on **distinct grid squares** — no two points share a square.
- If the LIS length is $\leq m$ and the LDS length is $\leq n$, all pairs land
  in an $m \times n$ grid with only $mn$ squares.
- But if we have $mn + 1$ elements in the sequence, the Pigeonhole Principle says
  two must share a square — a contradiction!
- Therefore, some element has $e_i \geq m+1$ (LIS of length $\geq m+1$) or
  $d_i \geq n+1$ (LDS of length $\geq n+1$). $\square$

## Section 9: 2D Sequences — Longest 2D-Increasing Subsequence

Can we generalise? A **2D sequence** is a sequence of points
$(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)$ in the plane.

A **2D-increasing subsequence** is one where both coordinates increase:
$x_{i_1} < x_{i_2} < \ldots$ and $y_{i_1} < y_{i_2} < \ldots$

This is equivalent to finding the longest chain in a **partial order**
on the points, where $(x, y) \prec (x', y')$ iff $x < x'$ AND $y < y'$.

In [ ]:
def longest_2d_increasing(points):
    """
    Find the longest 2D-increasing subsequence of a list of 2D points.
    A subsequence is 2D-increasing if BOTH coordinates increase strictly.

    Uses O(n^2) dynamic programming.

    Parameters
    ----------
    points : list of (x, y) tuples

    Returns
    -------
    best_subseq : list of (x, y) tuples
        One longest 2D-increasing subsequence.
    """
    n = len(points)
    # dp[i] = length of longest 2D-increasing subseq ending at point i
    dp = [1] * n
    parent = [-1] * n  # For reconstructing the subsequence

    for i in range(n):
        for j in range(i):
            xi, yi = points[i]
            xj, yj = points[j]
            # Can extend: both coords of point j are strictly less than point i
            if xj < xi and yj < yi:
                if dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
                    parent[i] = j

    # Find the endpoint of the longest subsequence
    best_end = max(range(n), key=lambda i: dp[i])

    # Reconstruct by following parent pointers backwards
    subseq = []
    idx = best_end
    while idx != -1:
        subseq.append(points[idx])
        idx = parent[idx]

    subseq.reverse()
    return subseq


# Generate a random 2D sequence
np.random.seed(77)
n_points = 20
points_2d = [(float(np.random.uniform(0, 100)),
              float(np.random.uniform(0, 100)))
             for _ in range(n_points)]

lis_2d = longest_2d_increasing(points_2d)

print(f'2D sequence with {n_points} points')
print(f'Longest 2D-increasing subsequence (length {len(lis_2d)}):')
for pt in lis_2d:
    print(f'  ({pt[0]:.1f}, {pt[1]:.1f})')

# Verify it's 2D-increasing
ok = all(
    lis_2d[k][0] < lis_2d[k+1][0] and lis_2d[k][1] < lis_2d[k+1][1]
    for k in range(len(lis_2d) - 1)
)
print(f'Verified 2D-increasing: {ok}')

In [ ]:
def visualise_2d_lis(points_2d, lis_2d):
    """
    Visualise the 2D sequence and highlight the longest 2D-increasing subsequence
    with arrows connecting consecutive elements.
    """
    lis_set = set(map(tuple, lis_2d))

    fig, ax = plt.subplots(figsize=(8, 7))

    # Plot all points
    for i, (x, y) in enumerate(points_2d):
        is_lis = (x, y) in lis_set
        color = 'tomato' if is_lis else 'steelblue'
        size  = 150 if is_lis else 60
        ax.scatter(x, y, color=color, s=size, zorder=5, edgecolors='black', linewidths=0.8)
        ax.annotate(f'$p_{{{i}}}$', (x, y), textcoords='offset points',
                    xytext=(5, 4), fontsize=8, color='black')

    # Draw arrows along the 2D-LIS
    for k in range(len(lis_2d) - 1):
        x1, y1 = lis_2d[k]
        x2, y2 = lis_2d[k+1]
        ax.annotate(
            '',
            xy=(x2, y2), xytext=(x1, y1),
            arrowprops=dict(
                arrowstyle='->', color='darkred',
                lw=2.5,
                connectionstyle='arc3,rad=0.1'
            )
        )

    # Shade the bounding box of the 2D-LIS to illustrate the ordering
    if lis_2d:
        xs_lis = [pt[0] for pt in lis_2d]
        ys_lis = [pt[1] for pt in lis_2d]

    ax.set_xlabel('x coordinate', fontsize=12)
    ax.set_ylabel('y coordinate', fontsize=12)
    ax.set_title(
        f'Longest 2D-Increasing Subsequence\n'
        f'Red points = 2D-LIS (length {len(lis_2d)}), '
        f'arrows show the ordering',
        fontsize=12
    )

    red_patch  = mpatches.Patch(color='tomato',    label=f'2D-LIS (length {len(lis_2d)})')
    blue_patch = mpatches.Patch(color='steelblue', label='Other points')
    ax.legend(handles=[red_patch, blue_patch], fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5, 110)
    ax.set_ylim(-5, 110)

    plt.tight_layout()
    plt.show()

    # Print the subsequence with arrows
    print('2D-LIS: ' + '  →  '.join(f'({x:.1f},{y:.1f})' for x, y in lis_2d))
    print('Both x and y coordinates are strictly increasing along the arrows.')


visualise_2d_lis(points_2d, lis_2d)

### Connection to Dilworth's Theorem

The problem of finding the longest 2D-increasing subsequence is closely related
to **Dilworth's Theorem** in order theory:

> In any finite partially ordered set, the maximum size of a chain
> equals the minimum number of antichains needed to cover the set.

Here, a **chain** is a 2D-increasing subsequence, and an **antichain** is a set
of points where no one is 2D-less than another (i.e. incomparable pairs).

This is a beautiful generalisation of the Erdős–Szekeres ideas to partial orders!

## Summary and Key Takeaways

In this notebook we explored **longest increasing subsequences** and the
**Erdős–Szekeres Theorem**.

### What We Learned

1. **LIS and LDS**: In any sequence, we can find the longest increasing (LIS)
   or decreasing (LDS) subsequence.

2. **Patience sorting**: A card-game algorithm that computes the LIS length.
   Number of piles = LIS length. Elegant and efficient!

3. **Erdős–Szekeres Theorem**: Any sequence of $> mn$ distinct numbers
   has an increasing subsequence of length $m+1$ or a decreasing one of length $n+1$.

4. **The $(e_i, d_i)$ labels**: Each position gets a pair (LIS-ending-here, LDS-ending-here).
   These pairs are always **distinct** — the key to the proof.

5. **Pigeonhole Principle**: If all pairs fit in an $m \times n$ grid but there
   are $mn+1$ elements, two elements must share a label — contradiction!

6. **Tight examples**: The grid construction gives sequences of length $n^2$ with
   no monotone subsequence of length $n+1$. The theorem is sharp!

7. **2D generalisation**: We can find the longest 2D-increasing chain of points,
   which connects to Dilworth's Theorem in order theory.

### Further Exploration

- The LIS problem has an $O(n \log n)$ algorithm using binary search on pile tops.
  Can you find it?
- Research the **Robinson–Schensted correspondence**: a deep connection between
  permutations and Young tableaux, which generalises patience sorting.
- The Erdős–Szekeres theorem has a beautiful 2D generalisation: the
  **Happy Ending Problem** (any 5 points in general position contain
  the vertices of a convex quadrilateral).